# LFW Grad-CAM — 00. Source and model freeze

완료된 aligned crop과 landmark bundle, 전체 LFW 범위, 검증된 ModelSpec을 새 immutable run에 고정합니다.

이 노트북은 한 단계만 실행하는 thin runbook입니다. 계산 구현은 `research/experiments/step4_workflow.py`에 있습니다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
EXECUTION = CONFIG["execution"]
MODEL_PROFILE = str(EXECUTION["model_profile"])
MODE = str(EXECUTION["mode"])
DATA_FRACTION = float(EXECUTION["data_fraction"])
EXECUTE_STAGE = bool(EXECUTION["execute_stage"])
WRITE_OUTPUTS = bool(EXECUTION["write_outputs"])
OVERWRITE = bool(EXECUTION["overwrite"])
DATASET_ID = "lfw"

if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")
if EXECUTE_STAGE and not WRITE_OUTPUTS:
    raise ValueError("정식 단계 실행은 WRITE_OUTPUTS=True여야 합니다.")

from research.experiments import freeze_step4_source_and_model


In [ ]:
if EXECUTE_STAGE:
    result = freeze_step4_source_and_model(
        CONFIG_PATH,
        project_root=PROJECT_ROOT,
        dataset_id=DATASET_ID,
        execution_acknowledged=True,
    )
else:
    result = {
        "dataset_id": DATASET_ID,
        "status": "not_executed",
        "reason": "CONFIG execution gates are closed",
    }

result


## 다음 단계

다음은 `01_origin_embedding_and_loo_templates.ipynb`입니다.

커널을 재시작한 뒤 다음 노트북을 위에서 아래로 실행합니다.